# 使用@tool装饰器来定义工具

## 1.自定义工具描述：description

举例1：

函数使用@tool装饰器修饰以后就是一个可以被模型识别的工具

如下的程序报错了，因为在没有提供description参数的情况下，要求函数必须提供docstring

In [ ]:
from accelerate.commands.config.update import description
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool
def get_weather(city: str):
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

修改为：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool
def get_weather(city: str):
    """获取城市的天气"""
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例2：使用description参数

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例3：

在同时声明了description和docstring的情况下：description的优先级更高

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    """获取城市的天气"""
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例4：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city:城市
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例5：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

# description的优先级是最高的
@tool(parse_docstring=True, description="获取具体城市的天气")
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city:城市
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

## 2.更改工具名称 name_or_callable（一般不会这样更改）

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(parse_docstring=True, name_or_callable="getWeather")
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city:城市
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

## 3.自定义args_schema

### 3.1 使用Pydantic模型定义

举例1：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint
from pydantic import BaseModel

class WeatherInput(BaseModel):
    city : str

@tool(args_schema=WeatherInput)
def get_weather(city: int):
    """
    获取城市的天气
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例2：

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint
from pydantic import BaseModel,Field

# 字段类型以BaseModel优先
class WeatherInput(BaseModel):
    city : str = Field(
        description="具体的城市",
        default="北京",
    )

@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    获取城市的天气
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

举例3：

In [40]:
from typing import Literal
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint
from pydantic import BaseModel,Field

class WeatherInput(BaseModel):
    city : str = Field(
        description="具体的城市",
        default="北京",
     )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default = "celsius",
        description="温度单位"
    )
    include_forecast: bool = Field(
        default = False,
        description="是否包含未来5天的天气预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = True):
    """
    获取城市的天气
    """
    return f"{city}阴雨连绵"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': '温度单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来5天的天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

## 3.2使用Json Schema定义

举例：

In [41]:
json_schema = {
            'properties': {
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': '温度单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来5天的天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = True):
    """
    获取城市的天气
    """
    return f"{city}阴雨连绵"